# Antenna Gain Pattern

We consider the directional antenna model defined in the 3GPP TR 38.901 standard.  
The function $G(\theta,\phi)$ represents the antenna gain as a function of the zenith angle $\theta$ and the azimuth angle $\phi$.

The model combines vertical and horizontal attenuation profiles together with a maximum directional gain. The resulting pattern defines a non-uniform distribution of radiated power over the unit sphere, concentrating energy around the boresight direction.

In [ ]:
import numpy as np

In [ ]:
def G_3gPP(theta, phi):
    """
    3GPP TR 38.901 antenna gain pattern.

    Parameters
    ----------
    theta : ndarray or float
        Zenith angle in radians.
        theta in [0, pi]

    phi : ndarray or float
        Azimuth angle in radians.
        phi in [-pi, pi]

    Returns
    -------
    gain : ndarray or float
        Linear gain (NOT dB)
    """

    theta_3db = np.deg2rad(65.0)
    phi_3db   = np.deg2rad(65.0)

    SLA_V = 30.0
    A_max = 30.0
    G_max = 8.0  # dBi

    # vertical attenuation
    A_v = -np.minimum(
        12.0 * ((theta - np.pi/2) / theta_3db) ** 2,
        SLA_V
    )

    # horizontal attenuation
    A_h = -np.minimum(
        12.0 * (phi / phi_3db) ** 2,
        A_max
    )

    # total attenuation + peak gain
    A_db = -np.minimum(
        -(A_v + A_h),
        A_max
    ) + G_max

    # linear gain
    return 10.0 ** (A_db / 10.0)

In [ ]:
import numpy as np
import plotly.graph_objects as go

# ============================================================
# Spherical grid
# ============================================================

n_theta = 200
n_phi = 400

theta = np.linspace(0, np.pi, n_theta)
phi   = np.linspace(-np.pi, np.pi, n_phi)

THETA, PHI = np.meshgrid(theta, phi)

GAIN = G_3gPP(THETA, PHI)


X = np.sin(THETA) * np.cos(PHI)
Y = np.sin(THETA) * np.sin(PHI)
Z = np.cos(THETA)

# ============================================================
# 3D plot
# ============================================================

fig = go.Figure(
    data=[
        go.Surface(
            x=X,
            y=Y,
            z=Z,
            surfacecolor=GAIN,
            colorscale="Plasma",   # violet -> yellow
            colorbar=dict(title="Gain"),
        )
    ]
)

fig.update_layout(
    title="Antenna Gain Map on the Unit Sphere",
    scene=dict(
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="z",
        aspectmode="data",
    ),
    width=600,
    height=600,
)

fig.show()

In [ ]:
import numpy as np

def sph_to_cart(theta, phi):
    """
    Vectorized spherical to Cartesian.
    Supports scalars and numpy arrays.
    """
    x = np.sin(theta) * np.cos(phi)
    y = np.sin(theta) * np.sin(phi)
    z = np.cos(theta)
    return np.stack([x, y, z], axis=0)  # (3, ...)


def cart_to_sph(v):
    """
    v: (3, ...) array
    """
    x, y, z = v
    norm = np.linalg.norm(v, axis=0) + 1e-12

    theta = np.arccos(np.clip(z / norm, -1.0, 1.0))
    phi = np.arctan2(y, x)

    return theta, phi


def rotation_matrix_z(phi):
    c, s = np.cos(phi), np.sin(phi)
    return np.array([
        [c, -s, 0],
        [s,  c, 0],
        [0,  0, 1]
    ])


def tilt_matrix_y(tilt):
    c, s = np.cos(tilt), np.sin(tilt)
    return np.array([
        [ c, 0, s],
        [ 0, 1, 0],
        [-s, 0, c]
    ])

def boresight_to_global(theta_tilt, phi_az):
    ct, st = np.cos(theta_tilt), np.sin(theta_tilt)
    tilt = np.array([
        [ ct, 0, st],
        [  0, 1,  0],
        [-st, 0, ct]
    ])

    ca, sa = np.cos(phi_az), np.sin(phi_az)
    az = np.array([
        [ca, -sa, 0],
        [sa,  ca, 0],
        [ 0,   0, 1]
    ])

    return az @ tilt




"""
Vectorized 3-sector antenna pattern.
theta, phi: arrays from meshgrid (same shape)
returns: gain array (same shape)
"""
def G(theta, phi):
	v = sph_to_cart(theta, phi)
	tilt = np.deg2rad(20)
	total = np.zeros(theta.shape, dtype=float)

	for k in range(3):
				az = k * 2*np.pi / 3
				Rt = tilt_matrix_y(tilt)
				Rz = rotation_matrix_z(az)
				R = Rz @ Rt
				v_flat = v.reshape(3, -1)
				v_local = R.T @ v_flat
				v_local = v_local.reshape(3, *theta.shape)

				th_loc, ph_loc = cart_to_sph(v_local)

				total += G_3gPP(th_loc, ph_loc)
	return total

To model a realistic cellular base station, we use a 3-sector antenna configuration with down-tilted beams. This reflects standard deployments in 3GPP cellular networks, where a single site is divided into three directional sectors covering the full azimuth (360°).

In [ ]:
import numpy as np
import plotly.graph_objects as go

# ============================================================
# Spherical grid
# ============================================================

n_theta = 200
n_phi = 400

theta = np.linspace(0, np.pi, n_theta)
phi   = np.linspace(-np.pi, np.pi, n_phi)

THETA, PHI = np.meshgrid(theta, phi)

GAIN = G(THETA, PHI)


X = np.sin(THETA) * np.cos(PHI)
Y = np.sin(THETA) * np.sin(PHI)
Z = np.cos(THETA)

# ============================================================
# 3D plot
# ============================================================

fig = go.Figure(
    data=[
        go.Surface(
            x=X,
            y=Y,
            z=Z,
            surfacecolor=GAIN,
            colorscale="Plasma",   # violet -> yellow
            colorbar=dict(title="Gain"),
        )
    ]
)

fig.update_layout(
    title="Antenna Gain Map on the Unit Sphere",
    scene=dict(
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="z",
        aspectmode="data",
    ),
    width=600,
    height=600,
)

fig.show()

## Step 1 — Random Sampling on the Sphere

As a first approximation, we generate a set of random directions on the unit sphere.  
This produces a fast and simple discretization of the angular domain before applying any geometric optimization procedure.

To obtain a uniform distribution over the sphere, we sample

$$
\phi \sim \mathcal{U}(-\pi,\pi)
$$

and

$$
u = \cos(\theta) \sim \mathcal{U}(-1,1).
$$

The spherical coordinates are then converted into Cartesian coordinates through

$$
x = \sin(\theta)\cos(\phi),
$$

$$
y = \sin(\theta)\sin(\phi),
$$

$$
z = \cos(\theta).
$$

Although this sampling is coarse and purely stochastic, it provides a computationally inexpensive initialization for subsequent refinement steps

In [ ]:
# Random sampling on the sphere

N = 250

phi = np.random.uniform(-np.pi, np.pi, N)
u   = np.random.uniform(-1.0, 1.0, N)

theta = np.arccos(u)

# Cartesian coordinates on the unit sphere
x = np.sin(theta) * np.cos(phi)
y = np.sin(theta) * np.sin(phi)
z = np.cos(theta)

In [ ]:
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=x,
            y=y,
            z=z,
            mode="markers",
            marker=dict(
                size=3,
                color=z,
                colorscale="Plasma",
                opacity=0.8,
            ),
        )
    ]
)

fig.update_layout(
    title="Random Sampling on the Unit Sphere",
    scene=dict(
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="z",
        aspectmode="data",
    ),
    width=400,
    height=400,
)

fig.show()

## Step 2 — Weighted Spherical Centroids

The objective is not to obtain a uniform discretization of the sphere, but rather a discretization adapted to the antenna gain distribution.

To achieve this, the gain map $G(\theta,\phi)$ is interpreted as a density function over the unit sphere:

$$
\rho(\theta,\phi) \propto G(\theta,\phi)
$$

Given a Voronoi region $V_i$, the associated centroid is defined as the density-weighted spherical centroid

$$
c_i =
\frac{
\int_{V_i} x\, G(x)\, d\omega(x)
}{
\left\|
\int_{V_i} x\, G(x)\, d\omega(x)
\right\|
}
$$

where $d\omega$ denotes the spherical surface measure.

Regions with larger antenna gain therefore attract more representative directions, producing a non-uniform discretization concentrated around high-gain areas.

In practice, exact integration over spherical Voronoi cells is computationally expensive for large-scale problems. Instead, we approximate the weighted centroids using Monte Carlo samples distributed on the sphere.

This approach is highly parallelizable and scales efficiently to large point sets, making it suitable for high-resolution antenna approximations.

In [ ]:
from sklearn.neighbors import KDTree
from joblib import Parallel, delayed
import numpy as np


# ============================================================
# Utilities
# ============================================================

def normalize(v):
    """
    Normalize vectors row-wise.
    """
    return v / np.linalg.norm(v, axis=-1, keepdims=True)


def cartesian_to_spherical(points):
    """
    Convert Cartesian coordinates to spherical angles.

    Returns
    -------
    theta : ndarray
        Zenith angle in [0, pi]

    phi : ndarray
        Azimuth angle in [-pi, pi]
    """

    x = points[:, 0]
    y = points[:, 1]
    z = points[:, 2]

    theta = np.arccos(np.clip(z, -1.0, 1.0))
    phi = np.arctan2(y, x)

    return theta, phi


# ============================================================
# Monte Carlo sampling on the sphere
# ============================================================

def sample_sphere(M):
    """
    Uniform random samples on S^2.
    """

    phi = np.random.uniform(-np.pi, np.pi, M)
    u   = np.random.uniform(-1.0, 1.0, M)

    theta = np.arccos(u)

    x = np.sin(theta) * np.cos(phi)
    y = np.sin(theta) * np.sin(phi)
    z = np.cos(theta)

    return np.column_stack([x, y, z])


# ============================================================
# Weighted spherical Lloyd iteration
# ============================================================

def weighted_spherical_lloyd_iteration(
    sites,
    M=200_000,
):
    """
    Perform one weighted spherical Lloyd iteration.

    Parameters
    ----------
    sites : ndarray of shape (N,3)
        Current Voronoi generators on S^2.

    M : int
        Number of Monte Carlo samples.

    Returns
    -------
    new_sites : ndarray of shape (N,3)
        Updated generators.
    """

    sites = normalize(sites)

    # --------------------------------------------------------
    # Monte Carlo samples
    # --------------------------------------------------------

    samples = sample_sphere(M)

    theta, phi = cartesian_to_spherical(samples)

    weights = G(theta, phi)**4

    # --------------------------------------------------------
    # Assign samples to nearest Voronoi site
    # --------------------------------------------------------

    tree = KDTree(sites)

    labels = tree.query(
        samples,
        k=1,
        return_distance=False,
    ).ravel()

    # --------------------------------------------------------
    # Weighted centroids
    # --------------------------------------------------------

    N = len(sites)

    accum = np.zeros((N, 3))
    mass  = np.zeros(N)

    np.add.at(
        accum,
        labels,
        samples * weights[:, None],
    )

    np.add.at(
        mass,
        labels,
        weights,
    )

    # Avoid empty cells
    empty = mass < 1e-12
    mass[empty] = 1.0

    new_sites = accum / mass[:, None]

    # Keep previous site if empty
    new_sites[empty] = sites[empty]

    # Project back to sphere
    new_sites = normalize(new_sites)

    return new_sites


# ============================================================
# Full weighted Lloyd algorithm
# ============================================================

def weighted_spherical_lloyd(
    sites,
    n_iterations=10,
    M=200_000,
):
    """
    Weighted spherical Lloyd relaxation.

    Parameters
    ----------
    sites : ndarray of shape (N,3)
        Initial directions.

    n_iterations : int
        Number of Lloyd iterations.

    M : int
        Monte Carlo samples per iteration.

    Returns
    -------
    sites : ndarray
        Relaxed sites.

    history : list
        Iteration history.
    """

    sites = normalize(sites)

    history = [sites.copy()]

    for _ in range(n_iterations):

        sites = weighted_spherical_lloyd_iteration(
            sites,
            M=M,
        )

        history.append(sites.copy())

    return sites, history

In [ ]:
# ============================================================
# Side-by-side visualization:
# Weighted spherical Lloyd vs gain map
# ============================================================

from plotly.subplots import make_subplots
import plotly.graph_objects as go

# ------------------------------------------------------------
# Initial random sites
# ------------------------------------------------------------

N_sites = 500

phi = np.random.uniform(-np.pi, np.pi, N_sites)
u   = np.random.uniform(-1.0, 1.0, N_sites)

theta = np.arccos(u)

x0 = np.sin(theta) * np.cos(phi)
y0 = np.sin(theta) * np.sin(phi)
z0 = np.cos(theta)

initial_sites = np.column_stack([x0, y0, z0])

# ------------------------------------------------------------
# Run weighted Lloyd
# ------------------------------------------------------------

points_final, history = weighted_spherical_lloyd(
    initial_sites,
    n_iterations=15,
    M=200_000,
)

# ------------------------------------------------------------
# Animation frames
# ------------------------------------------------------------

frames = []

for k, pts in enumerate(history):

    theta_pts, phi_pts = cartesian_to_spherical(pts)

    gain_pts = G(theta_pts, phi_pts)

    frame = go.Frame(
        data=[

            # Left: weighted discretization
            go.Scatter3d(
                x=pts[:, 0],
                y=pts[:, 1],
                z=pts[:, 2],
                mode="markers",
                marker=dict(
                    size=3,
                    color=gain_pts,
                    colorscale="Plasma",
                    opacity=0.9,
                    cmin=GAIN.min(),
                    cmax=GAIN.max(),
                ),
            ),

            # Right: continuous gain map
            go.Surface(
                x=X,
                y=Y,
                z=Z,
                surfacecolor=GAIN,
                colorscale="Plasma",
                showscale=False,
                cmin=GAIN.min(),
                cmax=GAIN.max(),
            ),

        ],
        name=str(k),
    )

    frames.append(frame)

# ------------------------------------------------------------
# Initial figure
# ------------------------------------------------------------

theta_init, phi_init = cartesian_to_spherical(history[0])

gain_init = G(theta_init, phi_init)

fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=(
        "Weighted Spherical Lloyd",
        "Continuous Gain Map",
    ),
)

# ------------------------------------------------------------
# Left subplot
# ------------------------------------------------------------

fig.add_trace(
    go.Scatter3d(
        x=history[0][:, 0],
        y=history[0][:, 1],
        z=history[0][:, 2],
        mode="markers",
        marker=dict(
            size=3,
            color=gain_init,
            colorscale="Plasma",
            opacity=0.9,
            cmin=GAIN.min(),
            cmax=GAIN.max(),
            colorbar=dict(title="Gain"),
        ),
    ),
    row=1,
    col=1,
)

# ------------------------------------------------------------
# Right subplot
# ------------------------------------------------------------

fig.add_trace(
    go.Surface(
        x=X,
        y=Y,
        z=Z,
        surfacecolor=GAIN,
        colorscale="Plasma",
        showscale=False,
        cmin=GAIN.min(),
        cmax=GAIN.max(),
    ),
    row=1,
    col=2,
)

fig.frames = frames

# ------------------------------------------------------------
# Layout
# ------------------------------------------------------------

fig.update_layout(
    width=800,
    height=400,
    updatemenus=[
        dict(
            type="buttons",
            showactive=False,
            buttons=[
                dict(
                    label="Play",
                    method="animate",
                    args=[
                        None,
                        dict(
                            frame=dict(duration=400, redraw=True),
                            transition=dict(duration=0),
                            fromcurrent=True,
                            mode="immediate",
                        ),
                    ],
                )
            ],
        )
    ],
)

# ------------------------------------------------------------
# Scene formatting
# ------------------------------------------------------------

fig.update_scenes(
    aspectmode="data",
    xaxis_title="x",
    yaxis_title="y",
    zaxis_title="z",
)

fig.show()

## Step 4 — Convergence Monitoring

To evaluate the convergence of the weighted spherical Lloyd algorithm, we measure the displacement of the generators between consecutive iterations.

Given the sets of directions

$$
\{p_i^{(k)}\}_{i=1}^N
\quad\text{and}\quad
\{p_i^{(k+1)}\}_{i=1}^N,
$$

we define the iteration error as the root mean square displacement

$$
E^{(k)}
=
\sqrt{
\frac{1}{N}
\sum_{i=1}^N
\left\|
p_i^{(k+1)} - p_i^{(k)}
\right\|^2
}.
$$

As the relaxation progresses, the generators stabilize and the error decreases toward zero.

This provides a simple quantitative criterion to monitor convergence of the discretization process.

In [ ]:
# ============================================================
# Convergence error
# ============================================================

errors = []

for k in range(len(history) - 1):

    p0 = history[k]
    p1 = history[k + 1]

    err = np.sqrt(
        np.mean(
            np.sum((p1 - p0)**2, axis=1)
        )
    )

    errors.append(err)

# ============================================================
# Error plot
# ============================================================

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=np.arange(1, len(errors) + 1),
        y=errors,
        mode="lines+markers",
    )
)

fig.update_layout(
    title="Weighted Spherical Lloyd Convergence",
    xaxis_title="Iteration",
    yaxis_title="RMS Displacement Error",
    width=700,
    height=400,
)

fig.show()

## Step 5 — Variational Energy of the Weighted CVT

The weighted spherical Lloyd algorithm can be interpreted as the minimization of a variational energy functional over the sphere.

Given Voronoi generators

$$
\{p_i\}_{i=1}^N \subset S^2,
$$

the associated weighted centroidal Voronoi tessellation minimizes the energy

$$
E(\{p_i\})
=
\sum_{i=1}^N
\int_{V_i}
\rho(x)\,
d(x,p_i)^2
\, d\omega(x),
$$

where:

- $V_i$ denotes the spherical Voronoi region associated with $p_i$,
- $\rho(x)$ is the target density,
- $d(\cdot,\cdot)$ is the spherical distance,
- $d\omega$ is the surface measure on $S^2$.

In this work, the density is induced by the antenna gain pattern:

$$
\rho(x) \propto G(x).
$$

Intuitively, the energy measures how well the discrete directions approximate the continuous gain distribution.

In [ ]:
# ============================================================
# Weighted CVT energy
# ============================================================

from sklearn.neighbors import KDTree
import plotly.graph_objects as go

# ------------------------------------------------------------
# Monte Carlo evaluation samples
# ------------------------------------------------------------

M_eval = 200_000

samples = sample_sphere(M_eval)

theta_s, phi_s = cartesian_to_spherical(samples)

rho = G(theta_s, phi_s)

energies = []

# ------------------------------------------------------------
# Evaluate energy at each iteration
# ------------------------------------------------------------

for pts in history:

    tree = KDTree(pts)

    distances, _ = tree.query(
        samples,
        k=1,
        return_distance=True,
    )

    distances = distances.ravel()

    # Euclidean squared distance on S^2
    energy = np.mean(
        rho * distances**2
    )

    energies.append(energy)

# ============================================================
# Energy plot
# ============================================================

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=np.arange(len(energies)),
        y=energies,
        mode="lines+markers",
    )
)

fig.update_layout(
    title="Weighted CVT Energy",
    xaxis_title="Iteration",
    yaxis_title="Energy",
    width=700,
    height=400,
)

fig.show()